# Nemotron 3.5 ASR Streaming 0.6B zero-shot baseline on OpenSLR 104 test

This notebook runs [`nvidia/nemotron-3.5-asr-streaming-0.6b`](https://huggingface.co/nvidia/nemotron-3.5-asr-streaming-0.6b) zero-shot on the OpenSLR 104 Hinglish test set (2,942 clips, 4.76 h) and scores it with the project normalizer and metrics in `src/hinglish_asr/`. It runs in Google Colab on an L4 GPU.

Nemotron 3.5 ASR is a cache-aware streaming FastConformer-RNNT model that supports Hindi (`hi-IN`). It is the main phone candidate: model 5 in the README is this zero-shot baseline, and model 6 is its Hinglish fine-tune.

**Streaming latency.** The model limits how far ahead each audio frame can attend (`num_lookahead_tokens`), and that limit sets the streaming latency. In `transformers`, a whole-clip pass applies the same attention limit as live streaming, so whole clips are transcribed in batches and the score equals streaming accuracy at that latency. The notebook runs two latencies:

- **560 ms** (`num_lookahead_tokens=6`): the largest setting within the project's under-1 s target.
- **1,120 ms** (`num_lookahead_tokens=13`): the model's most accurate setting.

- **Inputs:** the project folder on Google Drive, and `openslr104_test.zip` (the clips in `data/processed/openslr104/test/`).
- **Decoding:** `transformers` defaults (greedy RNN-T decoding), `language="auto"` so the model detects the language of each clip, float32 (the checkpoint's dtype), whole clips.
- **Outputs:** one row per latency in `results/summary.csv`, and each clip's output and detected language in `results/nemotron-3.5-asr-streaming-0.6b-zeroshot-<latency>ms/hyps.jsonl`.

Note: with the current scoring code T-WER equals WER, because `Normalizer.normalize` converts Devanagari loanwords to English before `t_wer` runs.

## Setup

Nemotron 3.5 ASR needs `transformers` 5.13 or newer. The install upgrades it only if Colab's version is older, and runs before anything imports `transformers`. If Colab asks to restart the session after the install, restart and continue from the next cell.

In [1]:
!pip install -q "transformers>=5.13.0" jiwer

In [2]:
from google.colab import drive

drive.mount("/content/drive")

PROJECT = "/content/drive/MyDrive/projects/asr"
ZIP_PATH = f"{PROJECT}/openslr104_test.zip"
AUDIO_DIR = "/content/openslr104_test"

Mounted at /content/drive


In [3]:
import os
import sys

import torch

os.chdir(PROJECT)
sys.path.insert(0, f"{PROJECT}/src")
print(torch.cuda.get_device_name(0))

NVIDIA L4


In [4]:
import zipfile

from tqdm.auto import tqdm

with zipfile.ZipFile(ZIP_PATH) as z:
    for member in tqdm(z.infolist(), desc="unzip"):
        z.extract(member, AUDIO_DIR)

unzip:   0%|          | 0/5886 [00:00<?, ?it/s]

## Test manifest

The manifest's `audio_path` values are local paths on the machine that made the data, so each clip is found by file name in the unzipped folder.

In [5]:
import json
from pathlib import Path

wavs = {p.name: str(p) for p in Path(AUDIO_DIR).rglob("*.wav")}
with open("data/processed/openslr104/test_manifest.jsonl", encoding="utf-8") as f:
    rows = [json.loads(line) for line in f]

missing = [r["utt_id"] for r in rows if Path(r["audio_path"]).name not in wavs]
assert not missing, f"{len(missing)} clips missing, e.g. {missing[:3]}"

paths = [wavs[Path(r["audio_path"]).name] for r in rows]
refs = [r["text"] for r in rows]
hours = sum(r["duration"] for r in rows) / 3600
print(f"{len(rows)} clips in manifest, {len(wavs)} wav files, {hours:.2f} h")
print(refs[0])
print(paths[0])

2942 clips in manifest, 5884 wav files, 4.76 h
libre office impress पर slide master और slide design के इस spoken tutorial में आपका स्वागत है
/content/openslr104_test/test/598753_406yMKxIdSDHRf8H_0000.wav


## Audio

In [6]:
import soundfile as sf

SR = 16000

audios = []
for path in tqdm(paths, desc="read audio"):
    audio, sr = sf.read(path, dtype="float32")
    assert sr == SR and audio.ndim == 1, (path, sr, audio.shape)
    audios.append(audio)

read audio:   0%|          | 0/2942 [00:00<?, ?it/s]

## Model

`processor.set_num_lookahead_tokens` sets the latency; the processor passes it to `generate` with the inputs. With `language="auto"` the model ends each transcript with a language tag such as `<hi-IN>`. It is a special token, so the text is decoded once without special tokens for scoring, and once with them to read the tag.

In [7]:
import re

from transformers import AutoModelForRNNT, AutoProcessor

MODEL = "nvidia/nemotron-3.5-asr-streaming-0.6b"
BATCH = 32

processor = AutoProcessor.from_pretrained(MODEL)
model = AutoModelForRNNT.from_pretrained(MODEL).to("cuda").eval()
print(model.dtype, processor.supported_streaming_latencies_ms)


def transcribe(audios):
    inputs = processor(audios, sampling_rate=SR, language="auto", return_tensors="pt")
    inputs = inputs.to(model.device, dtype=model.dtype)
    with torch.inference_mode():
        out = model.generate(**inputs, return_dict_in_generate=True)
    texts = processor.batch_decode(out.sequences, skip_special_tokens=True)
    raw = processor.batch_decode(out.sequences, skip_special_tokens=False)
    langs = [(re.findall(r"<([a-z]{2}-[A-Z]{2})>", r) or [""])[-1] for r in raw]
    return [t.strip() for t in texts], langs

processor_config.json:   0%|          | 0.00/2.52k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.38k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/881 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/752k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.55GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/655 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/193 [00:00<?, ?B/s]

torch.float32 {3: 320, 0: 80, 6: 560, 13: 1120}


A check on 5 clips at 560 ms before the full run. The detected language is shown in brackets.

In [8]:
processor.set_num_lookahead_tokens(6)
texts, langs = transcribe(audios[:5])
for ref, text, lang in zip(refs, texts, langs):
    print("REF:", ref)
    print(f"HYP ({lang}):", text, "\n")

/usr/local/lib/python3.13/dist-packages/transformers/generation/utils.py:1969: UserWarning: Using the model-agnostic default `max_length` (=1010) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


REF: libre office impress पर slide master और slide design के इस spoken tutorial में आपका स्वागत है
HYP (hi-IN): Liber office impress per Slide Master or Slide Design is Spoken tutorial में आपका स्वागत है। 

REF: इस tutorial में हम यह सीखेंगे कि slides के लिए backgrounds layout कैसे लागू करें
HYP (hi-IN): इस ट्यूटर में हम सीखेंगे कि स्लाइड्स के लिए बैकग्राउंड्स, लेआउट कैसे लागू करें।  यहां। 

REF: background slide पर लागू किये गये सभी रंगों और प्रभावों को उद्घृत करता है जो विषयवस्तु के पीछे मौजूद हैं
HYP (hi-IN): बैकग्राउंड स्लाइड पर लागू किए गए सभी रंगों और प्रभावों को उद्घित करता है जो विषय वस्तु के पीछे मौजूद है। 

REF: libre office impress में कई background options हैं जो बेहतर प्रस्तुतियाँ बनाने में आपकी मदद करते हैं
HYP (hi-IN): लिबर ऑफिस इम्प्रेस में कई बैकग्राउंड ऑप्शन्स हैं जो बेहतर प्रस्तुतियां बनाने में आपकी मदद। 

REF: आप भी अपने खुद़ के custom backgrounds बना सकते हैं
HYP (hi-IN): करते हैं।  आप भी अपने खुद के कस्टम बैकग्राउंड्स बना सकते हैं। 



## Transcribe the test set at both latencies

In [9]:
import time
from collections import Counter

runs = {}
for lookahead in (6, 13):
    processor.set_num_lookahead_tokens(lookahead)
    ms = processor.streaming_latency_ms
    texts, langs = [], []
    start = time.time()
    for b in tqdm(range(0, len(audios), BATCH), desc=f"{ms} ms"):
        t, l = transcribe(audios[b:b + BATCH])
        texts += t
        langs += l
    elapsed = time.time() - start
    runs[ms] = (texts, langs)

    print(f"{ms} ms: {len(texts)} outputs, {sum(not t for t in texts)} empty")
    print(f"{elapsed / 60:.1f} min, real-time factor {elapsed / (hours * 3600):.4f}")
    print("detected languages:", Counter(langs).most_common(), "\n")

560 ms:   0%|          | 0/92 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/transformers/generation/utils.py:1969: UserWarning: Using the model-agnostic default `max_length` (=1260) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/utils.py:1969: UserWarning: Using the model-agnostic default `max_length` (=1140) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/utils.py:1969: UserWarning: Using the model-agnostic default `max_length` (=1640) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/utils.py:1969: UserWarning: Using the model-agnostic default `max_length` (=2260) to

560 ms: 2942 outputs, 14 empty
2.6 min, real-time factor 0.0092
detected languages: [('hi-IN', 2603), ('', 333), ('en-US', 6)] 



1120 ms:   0%|          | 0/92 [00:00<?, ?it/s]

1120 ms: 2942 outputs, 20 empty
2.7 min, real-time factor 0.0094
detected languages: [('hi-IN', 2421), ('', 517), ('en-US', 4)] 



Each clip's output is saved before scoring, so the clips can be re-scored later without the GPU.

In [10]:
for ms, (texts, langs) in runs.items():
    out_dir = Path(f"results/nemotron-3.5-asr-streaming-0.6b-zeroshot-{ms}ms")
    out_dir.mkdir(parents=True, exist_ok=True)
    with open(out_dir / "hyps.jsonl", "w", encoding="utf-8") as f:
        for r, text, lang in zip(rows, texts, langs):
            f.write(json.dumps({"utt_id": r["utt_id"], "ref": r["text"], "hyp": text, "language": lang}, ensure_ascii=False) + "\n")

## Score

In [11]:
from hinglish_asr.metrics import score
from hinglish_asr.utils import Normalizer

norm = Normalizer("data/lists/loanwords.csv")
assert score(refs, refs, norm) == {"wer": 0.0, "t_wer": 0.0}

scores = {ms: score(refs, texts, norm) for ms, (texts, _) in runs.items()}
for ms, s in scores.items():
    print(f"{ms} ms: WER {s['wer']:.4f}  T-WER {s['t_wer']:.4f}")

560 ms: WER 0.4584  T-WER 0.4584
1120 ms: WER 0.4546  T-WER 0.4546


Normalized text that the metrics compare, for the first 5 clips at 560 ms:

In [12]:
for ref, hyp in zip(refs[:5], runs[560][0][:5]):
    print("REF:", norm.for_scoring(ref))
    print("HYP:", norm.for_scoring(hyp), "\n")

REF: libre office impress पर slide master और slide design के इस spoken tutorial में आपका स्वागत है
HYP: liber office impress per slide master or slide design is spoken tutorial में आपका स्वागत है 

REF: इस tutorial में हम यह सीखेंगे कि slides के लिए backgrounds layout कैसे लागू करें
HYP: इस tutor में हम सीखेंगे कि slides के लिए बैकग्राउंड्स layout कैसे लागू करें यहां 

REF: background slide पर लागू किये गये सभी रंगों और प्रभावों को उद्घृत करता है जो विषयवस्तु के पीछे मौजूद हैं
HYP: background slide पर लागू किए गए सभी रंगों और प्रभावों को उद्घित करता है जो विषय वस्तु के पीछे मौजूद है 

REF: libre office impress में कई background options हैं जो बेहतर प्रस्तुतियाँ बनाने में आपकी मदद करते हैं
HYP: libre office इम्प्रेस में कई background ऑप्शन्स हैं जो बेहतर प्रस्तुतियां बनाने में आपकी मदद 

REF: आप भी अपने खुद़ के custom backgrounds बना सकते हैं
HYP: करते हैं आप भी अपने खुद के custom बैकग्राउंड्स बना सकते हैं 



## Results file

In [13]:
import csv
import datetime

summary = Path("results/summary.csv")
write_header = not summary.exists()
with open(summary, "a", newline="", encoding="utf-8") as f:
    for ms, s in scores.items():
        row = {
            "model": MODEL,
            "dataset": "openslr104",
            "split": "test",
            "n_utts": len(rows),
            "wer": round(s["wer"], 4),
            "t_wer": round(s["t_wer"], 4),
            "decoding": f"zero-shot, greedy RNN-T, streaming latency {ms} ms, language=auto, {str(model.dtype).removeprefix('torch.')}, whole clips",
            "date": datetime.date.today().isoformat(),
        }
        writer = csv.DictWriter(f, fieldnames=row.keys())
        if write_header:
            writer.writeheader()
            write_header = False
        writer.writerow(row)

print(summary.read_text(encoding="utf-8"))

model,dataset,split,n_utts,wer,t_wer,decoding,date
openai/whisper-small,openslr104,test,2942,0.915,0.915,"zero-shot, temperature fallback 0.0-1.0 (compression ratio 1.35, logprob -1.0, seed 0), language=hi, fp16, 30 s chunks",2026-11-23
Trelis/tara,openslr104,test,2942,0.295,0.295,"zero-shot, temperature fallback 0.0-1.0 (compression ratio 1.35, logprob -1.0, seed 0), language=hi + mixedcode, bf16, 30 s pieces",2026-11-23
Qwen/Qwen3-ASR-0.6B,openslr104,test,2942,0.4362,0.4362,"zero-shot, qwen-asr 0.0.6 defaults (greedy, max 512 new tokens, repetition clean-up), language=None, bf16, whole clips",2026-11-23
nvidia/nemotron-3.5-asr-streaming-0.6b,openslr104,test,2942,0.4584,0.4584,"zero-shot, greedy RNN-T, streaming latency 560 ms, language=auto, float32, whole clips",2026-11-23
nvidia/nemotron-3.5-asr-streaming-0.6b,openslr104,test,2942,0.4546,0.4546,"zero-shot, greedy RNN-T, streaming latency 1120 ms, language=auto, float32, whole clips",2026-11-23

